# 02 - Model Experiments - Random Forest Regressor

Este notebook importa nuestro trainer oficial y efectúa Tuning de Hiperparámetros antes de mandar un modelo ciego a producción. Consumiremos las features generadas en el EDA.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import pandas as pd
from src.data.make_dataset import SalesTimeSerieExtractor
from src.features.build_features import build_preprocessing_pipeline, select_features_and_target
from src.training.train_sales_prediction import train_random_forest, evaluate_model, save_model

### Preparar Datos

In [ ]:
extractor = SalesTimeSerieExtractor()
try:
    df_raw = extractor.fetch_daily_sales()
    
    pipeline = build_preprocessing_pipeline()
    df_features = pipeline.fit_transform(df_raw)
    
    # Split 80/20
    train_size = int(len(df_features) * 0.8)
    df_train = df_features.iloc[:train_size]
    df_test = df_features.iloc[train_size:]
    
    X_train, y_train = select_features_and_target(df_train, 'y_sales_net')
    X_test, y_test = select_features_and_target(df_test, 'y_sales_net')
    print(f"X_Train: {X_train.shape}, Y_Train: {y_train.shape}")
except Exception as e:
    print("Error en conexión: ", e)

### Ejecutar Search CV

In [ ]:
if 'X_train' in locals() and not X_train.empty:
    # Activar el Tuner interno de preprocessor (RandomSearchCV + TimeSeriesSplit)
    model = train_random_forest(X_train, y_train, hyperparameter_search=True)
    
    # Test y métricas
    y_pred = model.predict(X_test)
    metrics = evaluate_model(y_test, y_pred)
    print("Resultados de Test de Validación Cruzada Temporal:")
    print(metrics)
    
    # Si las métricas nos agradan, guardamos los pkl en models/
    # save_model(model)